<a href="https://colab.research.google.com/github/aljeralmazan28-web/Algoverse-Research-TNAM/blob/main/DeepCDR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls /content
!ls /content/DeepCDR
!ls /content/DeepCDR/prog

DeepCDR  sample_data
checkpoint  DeepCDR_ECCB_presentation.pdf  model.png  README.md
data	    LICENSE			   prog
layers	  model.pyc	   run_DeepCDR_classify.py  utils.py
model.py  process_drug.py  run_DeepCDR.py


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cat /content/DeepCDR/prog/run_DeepCDR.py
!cat /content/DeepCDR/prog/model.py

import argparse
import random,os,sys
import numpy as np
import csv
from scipy import stats
import time
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import roc_auc_score
from sklearn import preprocessing
import pandas as pd
import keras.backend as K
from keras.models import Model, Sequential
from keras.models import load_model
from keras.layers import Input,InputLayer,Multiply,ZeroPadding2D
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Dense,Activation,Dropout,Flatten,Concatenate
from keras.layers import BatchNormalization
from keras.layers import Lambda
from keras import optimizers,utils
from keras.constraints import max_norm
from keras import regularizers
from keras.callbacks import ModelCheckpoint,Callback,EarlyStopping,History
from keras.utils import multi_gpu_model,plot_model
from keras.optimizers import Adam, SGD
from keras.models import model_from_json
import tensorflow as tf
from sklearn.metrics im

In [ ]:
!unzip -l "/content/drive/MyDrive/DeepCDR.zip" | head -50

Archive:  /content/drive/MyDrive/DeepCDR.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-10-25 22:21   DeepCDR/
        0  2026-10-26 00:34   DeepCDR/.git/
      338  2026-10-26 00:34   DeepCDR/.git/config
       73  2026-10-25 22:18   DeepCDR/.git/description
       23  2026-10-25 22:18   DeepCDR/.git/HEAD
        0  2026-10-25 22:18   DeepCDR/.git/hooks/
      478  2026-10-25 22:18   DeepCDR/.git/hooks/applypatch-msg.sample
     1972  2026-10-25 22:18   DeepCDR/.git/hooks/commit-msg.sample
     4611  2026-10-25 22:18   DeepCDR/.git/hooks/fsmonitor-watchman.sample
      189  2026-10-25 22:18   DeepCDR/.git/hooks/post-update.sample
      424  2026-10-25 22:18   DeepCDR/.git/hooks/pre-applypatch.sample
     1649  2026-10-25 22:18   DeepCDR/.git/hooks/pre-commit.sample
      416  2026-10-25 22:18   DeepCDR/.git/hooks/pre-merge-commit.sample
     1374  2026-10-25 22:18   DeepCDR/.git/hooks/pre-push.sample
     4898  2026-10-25 22:18   DeepCDR/.git

In [ ]:
!ls /content/DeepCDR/prog
!cat /content/DeepCDR/prog/run_DeepCDR.py

layers	  model.pyc	   run_DeepCDR_classify.py  utils.py
model.py  process_drug.py  run_DeepCDR.py
import argparse
import random,os,sys
import numpy as np
import csv
from scipy import stats
import time
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import roc_auc_score
from sklearn import preprocessing
import pandas as pd
import keras.backend as K
from keras.models import Model, Sequential
from keras.models import load_model
from keras.layers import Input,InputLayer,Multiply,ZeroPadding2D
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Dense,Activation,Dropout,Flatten,Concatenate
from keras.layers import BatchNormalization
from keras.layers import Lambda
from keras import optimizers,utils
from keras.constraints import max_norm
from keras import regularizers
from keras.callbacks import ModelCheckpoint,Callback,EarlyStopping,History
from keras.utils import multi_gpu_model,plot_model
from keras.optimizers import 

In [ ]:
p = '/content/DeepCDR/prog/run_DeepCDR.py'
s = open(p).read()

s = s.replace("from keras.utils import multi_gpu_model,plot_model", "from tensorflow.keras.utils import plot_model")
s = s.replace("import keras.backend as K", "import tensorflow.keras.backend as K")
s = s.replace("from keras", "from tensorflow.keras")
s = s.replace("print 'pcc-val: %s' % str(round(pcc_val,4))", "print('pcc-val: %s' % str(round(pcc_val,4)))")
s = s.replace("Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False)", "Adam(learning_rate=0.001)")
s = s.replace("np.Inf", "np.inf")
s = s.replace("nb_epoch=500", "nb_epoch=10")
s = s.replace("    data_train_idx,data_test_idx = DataSplit(data_idx)",
              "    data_idx = random.sample(data_idx, 3000)\n    print('subset size:', len(data_idx))\n    data_train_idx,data_test_idx = DataSplit(data_idx)")
if "\nmain()" not in s:
    s += "\n\nif __name__ == '__main__':\n    main()\n"

open(p, 'w').write(s)
print("Patch done")

Patch done


In [ ]:
!pip install -q hickle tf_keras

In [ ]:
p = '/content/DeepCDR/prog/layers/graph.py'
s = open(p).read()

s = s.replace("from keras.engine import Layer", "from tensorflow.keras.layers import Layer")
s = s.replace("import keras.backend as K", "import tensorflow.keras.backend as K")
s = s.replace("from keras import", "from tensorflow.keras import")
s = s.replace("\nimport keras\n", "\nimport tensorflow.keras as keras\n")
s = s.replace("return mask[0]", "return None if mask is None else mask[0]")

open(p, 'w').write(s)
print("Graph file fixed")

Graph file fixed


In [ ]:
%cd /content/DeepCDR/prog
!TF_USE_LEGACY_KERAS=1 python run_DeepCDR.py

/content/DeepCDR/prog
I0000 00:00:1792979827.485121    7933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792979827.557143    7933 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792979830.455883    7933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/content/DeepCDR/prog/layers/graph.py:67: SyntaxWarning: invalid escape sequence '\s'
  h_i^{(t)} = \sigma \left ( \frac{ G_i^T (h_i^{(t - 1)} W + b)}{\sum G_i}  \right )
Traceback (most recent call last):
  File "/content/DeepCDR/prog/run_DeepCDR.py", line 287, in <module>
    main()
    ~~~~^^
  File "/content/DeepCDR/prog/run_DeepCDR.py", line 267, in main
    mutation_feature, drug_feature,gexpr_feature,methylation_feature, dat

In [ ]:
!unzip -q -o /content/DeepCDR/data/GDSC/drug_graph_feat.zip -d /content/DeepCDR/data/GDSC/
!ls /content/DeepCDR/data/GDSC
!ls /content/DeepCDR/data/GDSC/drug_graph_feat | head -5

'1.Drug_listMon Jun 24 09_00_55 2019.csv'   drug_graph_feat.zip
 drug_graph_feat
10027278.hkl
10074640.hkl
10077147.hkl
10096043.hkl
10109823.hkl


In [ ]:
%cd /content/DeepCDR/prog
!TF_USE_LEGACY_KERAS=1 python run_DeepCDR.py

/content/DeepCDR/prog
I0000 00:00:1792979933.018968    8370 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792979933.141888    8370 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792979936.543951    8370 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/hickle/legacy_v3/hickle.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/usr/local/lib/python3.13/dist-packages/hickle/hickle.p

In [ ]:
import re
for f in ['run_DeepCDR.py', 'model.py', 'layers/graph.py']:
    p = '/content/DeepCDR/prog/' + f
    s = open(p).read()
    s = re.sub(r'^import (tensorflow\.)?keras\s*$', 'import tf_keras as keras', s, flags=re.M)
    s = re.sub(r'\bfrom (tensorflow\.)?keras', 'from tf_keras', s)
    s = re.sub(r'\bimport (tensorflow\.)?keras', 'import tf_keras', s)
    open(p, 'w').write(s)
print("All files updated")

All files updated


In [ ]:
%cd /content/DeepCDR/prog
!TF_USE_LEGACY_KERAS=1 python run_DeepCDR.py

/content/DeepCDR/prog
I0000 00:00:1792980070.593401    8938 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792980070.722992    8938 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792980074.563619    8938 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/content/DeepCDR/prog/layers/graph.py:67: SyntaxWarning: invalid escape sequence '\s'
  h_i^{(t)} = \sigma \left ( \frac{ G_i^T (h_i^{(t - 1)} W + b)}{\sum G_i}  \right )
/usr/local/lib/python3.13/dist-packages/hickle/legacy_v3/hickle.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain 

In [ ]:
import pandas as pd
df = pd.read_csv('/content/DeepCDR/data/CCLE/Cell_lines_annotations_20181226.txt', sep='\t')
print(df.columns.tolist())
col = [c for c in df.columns if 'gender' in c.lower() or 'sex' in c.lower()]
print("Sex column:", col)
if col:
    print(df[col[0]].value_counts(dropna=False))

['CCLE_ID', 'depMapID', 'Name', 'Pathology', 'Site_Primary', 'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3', 'Histology', 'Hist_Subtype1', 'Hist_Subtype2', 'Hist_Subtype3', 'Gender', 'Life_Stage', 'Age', 'Race', 'Geo_Loc', 'inferred_ethnicity', 'Site_Of_Finding', 'Disease', 'Annotation_Source', 'Original.Source.of.Cell.Line', 'Characteristics', 'Growth.Medium', 'Supplements', 'Freezing.Medium', 'Doubling.Time.from.Vendor', 'Doubling.Time.Calculated.hrs', 'type', 'type_refined', 'PATHOLOGIST_ANNOTATION', 'mutRate', 'tcga_code']
Sex column: ['Gender']
Gender
NaN       546
male      508
female    407
Name: count, dtype: int64


In [ ]:
p = '/content/DeepCDR/prog/run_DeepCDR.py'
s = open(p).read().replace("nb_epoch=10", "nb_epoch=3")
open(p, 'w').write(s)

15247

In [ ]:
%cd /content/DeepCDR/prog
!TF_USE_LEGACY_KERAS=1 python run_DeepCDR.py

/content/DeepCDR/prog
I0000 00:00:1792981395.808118   14301 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792981395.952634   14301 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792981399.355407   14301 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/hickle/legacy_v3/hickle.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/usr/local/lib/python3.13/dist-packages/hickle/hickle.p

In [ ]:
import torch, numpy, pandas, sklearn
print("torch==" + torch.__version__.split("+")[0])
print("numpy==" + numpy.__version__)
print("pandas==" + pandas.__version__)
print("scikit-learn==" + sklearn.__version__)

torch==2.11.0
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1


In [ ]:
!pip install -q torch==2.11.0 numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1